# PyRadiomics feature extraction: one CSV per filter

| file | filter module | PyRadiomics image type(s) |
|---|---|---|
| `features_raw_image.csv` | `raw_image.py` | `Original` (+ shape features) |
| `features_binary_patterns.csv` | `binary_patterns.py` | `LBP3D` |
| `features_gradient_filter.csv` | `gradient_filter.py` | `Gradient` |
| `features_log_filter.csv` | `log_filter.py` | `LoG` (σ = 1–5 mm) |
| `features_wavelet_filter.csv` | `wavelet_filter.py` | `Wavelet` (8 coif1 sub-bands) |
| `features_square_filter.csv` | `square_filter.py` | `Square` |
| `features_square_root_filter.csv` | `square_root_filter.py` | `SquareRoot` |
| `features_logarithmic_filter.csv` | `logarithmic_filter.py` | `Logarithm` |
| `features_exponential_filter.csv` | `exponential_filter.py` | `Exponential` |
| `features_gabor_filter.csv` | `gabor_filter.py` | `Gabor` (custom, IBSI 2) |
| `features_laws_filter.csv` | `laws_filter.py` | `Laws` (custom, IBSI 2) |

Each CSV has one row per task (case × side × channel) with the columns `case, side, channel, task` followed by
`<image type>_<feature class>_<feature>` (PyRadiomics naming, e.g. `log-sigma-2-0-mm-3D_glcm_Contrast`).
Shape features depend only on the mask, so they are stored once, in `features_raw_image.csv`.

**How:** for every task the image is pre-processed **once** by PyRadiomics (normalize → resample → crop → mask
check), then each filter generator is run and its images are passed to PyRadiomics' feature classes, exactly
as `RadiomicsFeatureExtractor.execute()` does. Gabor and Laws are not part of PyRadiomics 3.1. They are
registered as extra image types (`radiomics.imageoperations.getGaborImage` / `getLawsImage`), so the same
code path is used.

Settings come from `notebooks/setup.yaml → PARAMS.setting` (shared with the other notebooks); filter
parameters, feature classes and output folders from `pyradiomics_params.yaml`.

**Kernel:** run with the `pyrad39` conda env (Python 3.9 + PyRadiomics 3.1.0).

In [1]:
import sys
import logging
from pathlib import Path

# folders of the pipeline code, so that its modules can be imported from this notebook
CODE_ROOT = Path("/Users/kamleshranabhat/Desktop/radiomics/full_pipeline")
for folder in ["", "scripts", "pre_processing", "feature_extraction", "post_processing"]:
    if str(CODE_ROOT / folder) not in sys.path:
        sys.path.insert(0, str(CODE_ROOT / folder))
import env_fixes                               # pyrad39 work-arounds (MKL crash, pathlib mkdir); before numpy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import radiomics
from IPython.display import display

radiomics.setVerbosity(logging.ERROR)          # hide the 'fixed bin count' warnings
from pipeline import *                         # also registers the custom Gabor / Laws image types

params = load_params()
print("PyRadiomics", radiomics.__version__, "| image types:", radiomics.getImageTypes())

PyRadiomics v3.1.0 | image types: ['Exponential', 'Gabor', 'Gradient', 'LBP2D', 'LBP3D', 'Laws', 'LoG', 'Logarithm', 'Original', 'Square', 'SquareRoot', 'Wavelet']


## Settings

In [2]:
print("pre-processing options:", params["options"])
print("feature classes:", params["featureClass"], "| shape in raw_image:", params["shape_in_raw_image"])
print("output:", params["feature_dir"])
display(pd.Series(params["setting"], name="value").to_frame())
display(filter_table(params))

pre-processing options: {'registration': True, 'bias_correction': False}
feature classes: ['firstorder', 'glcm', 'glrlm', 'glszm', 'gldm', 'ngtdm'] | shape in raw_image: True
output: C:\Users\krana\Desktop\radiomics\full_pipeline\outputs\features\pyradiomics


,value
normalize,True
normalizeScale,100
removeOutliers,3
voxelArrayShift,300
resampledPixelSpacing,"[1, 1, 1]"
interpolator,sitkBSpline
padDistance,10
preCrop,True
binCount,32
label,1


,filter,csv,pyradiomics_image_type,custom,enabled,args
0,raw_image,features_raw_image.csv,Original,False,True,{}
1,binary_patterns,features_binary_patterns.csv,LBP3D,False,True,"{'lbp3DLevels': 2, 'lbp3DIcosphereRadius': 1, ..."
2,gradient_filter,features_gradient_filter.csv,Gradient,False,True,{'gradientUseSpacing': True}
3,log_filter,features_log_filter.csv,LoG,False,True,"{'sigma': [1.0, 2.0, 3.0, 4.0, 5.0]}"
4,wavelet_filter,features_wavelet_filter.csv,Wavelet,False,True,"{'wavelet': 'coif1', 'level': 1, 'start_level'..."
5,square_filter,features_square_filter.csv,Square,False,True,{}
6,square_root_filter,features_square_root_filter.csv,SquareRoot,False,True,{}
7,logarithmic_filter,features_logarithmic_filter.csv,Logarithm,False,True,{}
8,exponential_filter,features_exponential_filter.csv,Exponential,False,True,{}
9,gabor_filter,features_gabor_filter.csv,Gabor,True,True,"{'gaborSigma': [2.0, 4.0], 'gaborLambda': [4.0..."


## Tasks
Every case × breast side × MRI channel. To test on a subset, slice the list (e.g. `tasks[:2]`).

In [3]:
tasks = all_tasks()
print(len(tasks), "tasks, e.g.", [task_name(*t) for t in tasks[:4]])

40 tasks, e.g. ['BreastMRI002_L_ch0', 'BreastMRI002_L_ch1', 'BreastMRI002_L_ch2', 'BreastMRI002_L_ch3']


## Check against the PyRadiomics 3.1.0 reference run
The built-in filters are re-run for one task with `REFERENCE_OPTIONS` (no registration, as in the reference
run in `dataset/derived/features`) and compared feature by feature. Every feature should match to ~1e-12.

In [4]:
check = compare_with_reference(("BreastMRI023", "L", 0), params)
per_filter = check.groupby("filter").agg(features=("feature", "size"),
                                         not_in_reference=("reference", lambda s: int(s.isna().sum())),
                                         max_relative_error=("rel_error", "max"))
display(per_filter)
summarize_comparison(check, tolerance=1e-9)

,features,not_in_reference,max_relative_error
filter,,,
binary_patterns,279,0,0.000000e+00
exponential_filter,93,0,0.000000e+00
gradient_filter,93,0,0.000000e+00
log_filter,465,0,0.000000e+00
logarithmic_filter,93,0,0.000000e+00
raw_image,107,0,3.749475e-16
square_filter,93,0,0.000000e+00
square_root_filter,93,0,0.000000e+00
wavelet_filter,744,0,0.000000e+00


2060/2060 features within relative error 1e-09 of PyRadiomics (largest relative error 3.75e-16)


## Extract all tasks
Results are cached per task in `outputs/features/pyradiomics/tasks/<task>.json`. If the run is interrupted, re-run
the cell and finished tasks are skipped. A cached filter is recomputed automatically when its settings change.
Use `resume=False` to force a full recomputation. Expect roughly 20–30 s per task.

In [5]:
with Timer() as timer:
    tables = run_extraction(tasks, params, resume=True)

[1/40] BreastMRI002_L_ch0: 2990 features in 38 s (raw_image 9s, binary_patterns 4s, gradient_filter 1s, log_filter 3s, wavelet_filter 5s, square_filter 1s, square_root_filter 1s, logarithmic_filter 1s, exponential_filter 1s, gabor_filter 8s, laws_filter 5s)


[2/40] BreastMRI002_L_ch1: 2990 features in 37 s (raw_image 9s, binary_patterns 4s, gradient_filter 1s, log_filter 3s, wavelet_filter 5s, square_filter 1s, square_root_filter 1s, logarithmic_filter 1s, exponential_filter 1s, gabor_filter 8s, laws_filter 5s)


[3/40] BreastMRI002_L_ch2: 2990 features in 37 s (raw_image 9s, binary_patterns 4s, gradient_filter 1s, log_filter 3s, wavelet_filter 5s, square_filter 1s, square_root_filter 1s, logarithmic_filter 1s, exponential_filter 1s, gabor_filter 8s, laws_filter 5s)


[4/40] BreastMRI002_L_ch3: 2990 features in 37 s (raw_image 9s, binary_patterns 4s, gradient_filter 1s, log_filter 3s, wavelet_filter 5s, square_filter 1s, square_root_filter 1s, logarithmic_filter 1s, exponential_filter 1s, gabor_filter 8s, laws_filter 5s)


[5/40] BreastMRI002_R_ch0: 2990 features in 35 s (raw_image 10s, binary_patterns 4s, gradient_filter 0s, log_filter 2s, wavelet_filter 4s, square_filter 1s, square_root_filter 1s, logarithmic_filter 1s, exponential_filter 1s, gabor_filter 7s, laws_filter 5s)


...

[36/40] BreastMRI023_L_ch3: 2990 features in 21 s (raw_image 6s, binary_patterns 1s, gradient_filter 0s, log_filter 1s, wavelet_filter 2s, square_filter 0s, square_root_filter 0s, logarithmic_filter 0s, exponential_filter 0s, gabor_filter 5s, laws_filter 3s)


[37/40] BreastMRI023_R_ch0: 2990 features in 44 s (raw_image 18s, binary_patterns 3s, gradient_filter 0s, log_filter 3s, wavelet_filter 4s, square_filter 1s, square_root_filter 1s, logarithmic_filter 1s, exponential_filter 1s, gabor_filter 8s, laws_filter 5s)


[38/40] BreastMRI023_R_ch1: 2990 features in 45 s (raw_image 18s, binary_patterns 3s, gradient_filter 0s, log_filter 3s, wavelet_filter 4s, square_filter 1s, square_root_filter 1s, logarithmic_filter 1s, exponential_filter 1s, gabor_filter 9s, laws_filter 6s)


[39/40] BreastMRI023_R_ch2: 2990 features in 45 s (raw_image 19s, binary_patterns 3s, gradient_filter 0s, log_filter 3s, wavelet_filter 5s, square_filter 1s, square_root_filter 1s, logarithmic_filter 1s, exponential_filter 1s, gabor_filter 9s, laws_filter 5s)


[40/40] BreastMRI023_R_ch3: 2990 features in 45 s (raw_image 19s, binary_patterns 3s, gradient_filter 0s, log_filter 3s, wavelet_filter 4s, square_filter 1s, square_root_filter 1s, logarithmic_filter 1s, exponential_filter 1s, gabor_filter 8s, laws_filter 5s)
Time taken: 1656.4 seconds


## Output tables

In [6]:
summary = summarize_tables(tables)
summary["size_kB"] = [round((Path(params["feature_dir"]) / f).stat().st_size / 1024, 1) for f in summary["file"]]
display(summary)

,file,rows,features,image_types,glcm,firstorder,glrlm,glszm,shape,gldm,ngtdm,missing_values,size_kB
0,features_raw_image.csv,40,107,1,24,18,16,16,14.0,14,5,0,83.5
1,features_binary_patterns.csv,40,279,3,72,54,48,48,0.0,42,15,0,216.4
2,features_gradient_filter.csv,40,93,1,24,18,16,16,0.0,14,5,0,73.1
3,features_log_filter.csv,40,465,5,120,90,80,80,0.0,70,25,0,367.4
4,features_wavelet_filter.csv,40,744,8,192,144,128,128,0.0,112,40,0,581.1
5,features_square_filter.csv,40,93,1,24,18,16,16,0.0,14,5,0,73.1
6,features_square_root_filter.csv,40,93,1,24,18,16,16,0.0,14,5,0,73.4
7,features_logarithmic_filter.csv,40,93,1,24,18,16,16,0.0,14,5,0,73.4
8,features_exponential_filter.csv,40,93,1,24,18,16,16,0.0,14,5,0,73.3
9,features_gabor_filter.csv,40,372,4,96,72,64,64,0.0,56,20,0,300.0


In [7]:
seconds = pd.DataFrame([json.loads(task_cache_path(params, t).read_text())["seconds"]
                        for t in tasks if task_cache_path(params, t).exists()])
display(seconds.describe().loc[["mean", "min", "max"]].round(1))

,raw_image,binary_patterns,gradient_filter,log_filter,wavelet_filter,square_filter,square_root_filter,logarithmic_filter,exponential_filter,gabor_filter,laws_filter
mean,14.3,3.1,0.5,2.3,4.1,0.5,0.5,0.5,0.5,7.6,4.6
min,6.3,1.4,0.2,1.1,1.9,0.2,0.2,0.2,0.2,4.5,2.4
max,34.9,5.3,0.7,3.3,5.7,0.7,0.8,0.8,0.7,10.6,6.5


## Reading the tables back
Example: load one table and look at a few LoG features per channel.

In [8]:
log_table = pd.read_csv(Path(params["feature_dir"]) / "features_log_filter.csv")
columns = ["task"] + [c for c in log_table.columns if c.endswith("firstorder_Mean") or c.endswith("glcm_Contrast")][:4]
display(log_table[columns].head(8))

,task,log-sigma-1-0-mm-3D_firstorder_Mean,log-sigma-1-0-mm-3D_glcm_Contrast,log-sigma-2-0-mm-3D_firstorder_Mean,log-sigma-2-0-mm-3D_glcm_Contrast
0,BreastMRI002_L_ch0,-8.295261,9.495389,-16.770613,4.654112
1,BreastMRI002_L_ch1,-8.568654,8.053580,-16.743632,4.349957
2,BreastMRI002_L_ch2,-10.178296,6.294211,-18.608184,3.192961
3,BreastMRI002_L_ch3,-9.036696,8.154111,-17.302232,4.435670
4,BreastMRI002_R_ch0,-12.544707,9.205640,-18.424656,5.270092
5,BreastMRI002_R_ch1,-10.681183,9.103317,-17.385837,4.703219
6,BreastMRI002_R_ch2,-11.274357,7.686866,-19.091261,4.106580
7,BreastMRI002_R_ch3,-11.505006,9.394665,-17.968028,4.775576
